In [13]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt


In [14]:
data = pd.read_csv("train.csv")
print(data.head())
print(data.info())


   sample_id                                    catalog_content  \
0      33127  Item Name: La Victoria Green Taco Sauce Mild, ...   
1     198967  Item Name: Salerno Cookies, The Original Butte...   
2     261251  Item Name: Bear Creek Hearty Soup Bowl, Creamy...   
3      55858  Item Name: Judee’s Blue Cheese Powder 11.25 oz...   
4     292686  Item Name: kedem Sherry Cooking Wine, 12.7 Oun...   

                                          image_link  price  
0  https://m.media-amazon.com/images/I/51mo8htwTH...   4.89  
1  https://m.media-amazon.com/images/I/71YtriIHAA...  13.12  
2  https://m.media-amazon.com/images/I/51+PFEe-w-...   1.97  
3  https://m.media-amazon.com/images/I/41mu0HAToD...  30.34  
4  https://m.media-amazon.com/images/I/41sA037+Qv...  66.49  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sample_id        750

In [16]:
"""
LightGBM pipeline for ML Challenge 2025 - Smart Product Pricing
- Input: /mnt/data/train.csv (or ./train.csv)
- Output: saved model artifacts in ./model_artifacts and optional sample_model_test_out.csv (if ./test.csv exists)

Configure SAMPLE_SIZE for fast iterations (use a smaller subset), set None to use all data.
"""

import os
import re
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# ------------------ USER CONFIG ------------------
TRAIN_CSV = "train.csv"     # path to your train.csv
TEST_CSV = "test.csv"       # optional path to test.csv
OUT_DIR = "model_artifacts"
SAMPLE_SIZE = 20000         # set None to train on full dataset (be careful: can be slow)
TFIDF_MAX_FEATURES = 300    # reduce for faster experiments; increase for final runs
RANDOM_STATE = 42
# -------------------------------------------------

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0 + 1e-9
    return np.mean(np.abs(y_pred - y_true) / denom) * 100.0

# --- Load
df = pd.read_csv(TRAIN_CSV)
print("Loaded:", df.shape)
assert 'price' in df.columns, "train.csv must contain 'price' column"

# --- Simple feature engineering from catalog_content
def extract_ipq(text):
    text_low = text.lower()
    patterns = [r'pack of\s*(\d+)', r'(\d+)\s*pack', r'(\d+)[- ]?count', r'(\d+)\s*pcs', r'(\d+)\s*piece']
    for p in patterns:
        m = re.search(p, text_low)
        if m:
            try:
                v = int(m.group(1))
                if 1 <= v <= 10000:
                    return v
            except:
                pass
    nums = re.findall(r'\b(\d{1,3})\b', text_low)
    for n in nums:
        v = int(n)
        if 1 < v <= 1000:
            return v
    return 1

def extract_brand(text):
    txt = text.strip()
    if not txt: return 'unknown'
    m = re.match(r'^([^:\-–—|,]+?)\s*[\-:|,]', txt)
    if m: return m.group(1).strip().lower()
    m2 = re.search(r'by\s+([A-Za-z0-9& ]{2,30})', txt, flags=re.IGNORECASE)
    if m2: return m2.group(1).strip().lower()
    return txt.split()[0].lower()

df['ipq'] = df['catalog_content'].apply(extract_ipq)
df['char_count'] = df['catalog_content'].apply(len)
df['word_count'] = df['catalog_content'].apply(lambda x: len(x.split()))
df['digit_count'] = df['catalog_content'].apply(lambda x: sum(c.isdigit() for c in x))
df['upper_count'] = df['catalog_content'].apply(lambda x: sum(1 for c in x if c.isupper()))
df['has_image'] = df['image_link'].notna() & (df['image_link'].str.strip() != '')
df['brand_guess'] = df['catalog_content'].apply(extract_brand)

# --- Target and log-transform
df = df[df['price'] > 0].copy()
y = df['price'].values
y_log = np.log1p(y)

# --- Text features (TF-IDF)
tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1,2), min_df=5)
X_text = tfidf.fit_transform(df['catalog_content'])

# --- Numeric features scaling
num_cols = ['ipq','char_count','word_count','digit_count','upper_count','has_image']
X_num = df[num_cols].astype(float).values
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

# --- Brand frequency (simple freq encoding)
brand_counts = df['brand_guess'].value_counts().to_dict()
df['brand_freq'] = df['brand_guess'].map(lambda x: brand_counts.get(x,0))
brand_scaler = StandardScaler()
X_brand_scaled = brand_scaler.fit_transform(df[['brand_freq']].astype(float).values)

# --- Combine into sparse matrix
from scipy.sparse import hstack, csr_matrix
X_all = hstack([X_text, csr_matrix(X_num_scaled), csr_matrix(X_brand_scaled)], format='csr')
print("Feature matrix:", X_all.shape)

# --- Split
X_train, X_val, y_train_log, y_val_log = train_test_split(X_all, y_log, test_size=0.2, random_state=RANDOM_STATE)

# --- Train LightGBM on log-target
model = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.1,
    num_leaves=256,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    eval_metric='rmse',
    callbacks=[
        # early stopping
        __import__('lightgbm').early_stopping(stopping_rounds=50),
        # reduce logging frequency
        __import__('lightgbm').log_evaluation(period=100)
    ]
)

# --- Predict back to original space
y_val_pred_log = model.predict(X_val)
y_val_pred = np.expm1(y_val_pred_log)
y_val_true = np.expm1(y_val_log)

rmse = np.sqrt(mean_squared_error(y_val_true, y_val_pred))
print("Validation RMSE:", rmse)
print("Validation SMAPE:", smape(y_val_true, y_val_pred))

# --- Save artifacts
os.makedirs(OUT_DIR, exist_ok=True)
joblib.dump(model, os.path.join(OUT_DIR, "lgbm_model.joblib"))
joblib.dump(tfidf, os.path.join(OUT_DIR, "tfidf_vectorizer.joblib"))
joblib.dump(scaler, os.path.join(OUT_DIR, "scaler.joblib"))
joblib.dump(brand_scaler, os.path.join(OUT_DIR, "brand_scaler.joblib"))
print("Saved artifacts to", OUT_DIR)

# --- If test.csv exists, create a submission
if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    test_df['catalog_content'] = test_df['catalog_content'].fillna('').astype(str)
    test_df['ipq'] = test_df['catalog_content'].apply(extract_ipq)
    test_df['char_count'] = test_df['catalog_content'].apply(len)
    test_df['word_count'] = test_df['catalog_content'].apply(lambda x: len(x.split()))
    test_df['digit_count'] = test_df['catalog_content'].apply(lambda x: sum(c.isdigit() for c in x))
    test_df['upper_count'] = test_df['catalog_content'].apply(lambda x: sum(1 for c in x if c.isupper()))
    test_df['has_image'] = test_df.get('image_link', '').notna() & (test_df.get('image_link', '').str.strip() != '')
    test_df['brand_guess'] = test_df['catalog_content'].apply(extract_brand)
    test_df['brand_freq'] = test_df['brand_guess'].map(lambda x: brand_counts.get(x,0))
    X_test_text = tfidf.transform(test_df['catalog_content'])
    X_test_num = scaler.transform(test_df[num_cols].astype(float).values)
    X_test_brand = brand_scaler.transform(test_df[['brand_freq']].astype(float).values)
    X_test_all = hstack([X_test_text, csr_matrix(X_test_num), csr_matrix(X_test_brand)], format='csr')
    y_test_log_pred = model.predict(X_test_all)
    y_test_pred = np.expm1(y_test_log_pred)
    out = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': y_test_pred})
    out.to_csv('sample_model_test_out.csv', index=False)
    print("Saved sample_model_test_out.csv")


Loaded: (75000, 4)
Feature matrix: (75000, 307)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.247423 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 77572
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 305
[LightGBM] [Info] Start training from score 2.740904
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.73369	valid_0's l2: 0.538301
[200]	valid_0's rmse: 0.730992	valid_0's l2: 0.534349
Early stopping, best iteration is:
[190]	valid_0's rmse: 0.730722	valid_0's l2: 0.533955


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Validation RMSE: 34.39012154280988
Validation SMAPE: 55.08750896776432
Saved artifacts to model_artifacts


In [19]:
import joblib, pandas as pd, numpy as np
from scipy.sparse import hstack, csr_matrix
import os

# Define the output directory
OUT_DIR = "model_artifacts"
TEST_CSV = "test.csv"

# Load the saved artifacts
model = joblib.load(os.path.join(OUT_DIR, "lgbm_model.joblib"))
tfidf = joblib.load(os.path.join(OUT_DIR, "tfidf_vectorizer.joblib"))
scaler = joblib.load(os.path.join(OUT_DIR, "scaler.joblib"))
brand_scaler = joblib.load(os.path.join(OUT_DIR, "brand_scaler.joblib"))

# Load the test data
test_df = pd.read_csv(TEST_CSV)

# Recreate preprocessing steps on test data (same as in cell 6A_KUTaaK7nW)
def extract_ipq(text):
    text_low = text.lower()
    patterns = [r'pack of\s*(\d+)', r'(\d+)\s*pack', r'(\d+)[- ]?count', r'(\d+)\s*pcs', r'(\d+)\s*piece']
    for p in patterns:
        m = re.search(p, text_low)
        if m:
            try:
                v = int(m.group(1))
                if 1 <= v <= 10000:
                    return v
            except:
                pass
    nums = re.findall(r'\b(\d{1,3})\b', text_low)
    for n in nums:
        v = int(n)
        if 1 < v <= 1000:
            return v
    return 1

def extract_brand(text):
    txt = text.strip()
    if not txt: return 'unknown'
    m = re.match(r'^([^:\-–—|,]+?)\s*[\-:|,]', txt)
    if m: return m.group(1).strip().lower()
    m2 = re.search(r'by\s+([A-Za-z0-9& ]{2,30})', txt, flags=re.IGNORECASE)
    if m2: return m2.group(1).strip().lower()
    return txt.split()[0].lower()

test_df['catalog_content'] = test_df['catalog_content'].fillna('').astype(str)
test_df['ipq'] = test_df['catalog_content'].apply(extract_ipq)
test_df['char_count'] = test_df['catalog_content'].apply(len)
test_df['word_count'] = test_df['catalog_content'].apply(lambda x: len(x.split()))
test_df['digit_count'] = test_df['catalog_content'].apply(lambda x: sum(c.isdigit() for c in x))
test_df['upper_count'] = test_df['catalog_content'].apply(lambda x: sum(1 for c in x if c.isupper()))
test_df['has_image'] = test_df.get('image_link', '').notna() & (test_df.get('image_link', '').str.strip() != '')
test_df['brand_guess'] = test_df['catalog_content'].apply(extract_brand)

# Use the brand_counts from the training data to map brand frequencies in the test data
brand_counts = {brand: count for brand, count in zip(df['brand_guess'], df['brand_freq'])} # Assuming df is available from previous cell
test_df['brand_freq'] = test_df['brand_guess'].map(lambda x: brand_counts.get(x, 0))


X_test_text = tfidf.transform(test_df['catalog_content'])
num_cols = ['ipq','char_count','word_count','digit_count','upper_count','has_image']
X_test_num = scaler.transform(test_df[num_cols].astype(float).values)
X_test_brand = brand_scaler.transform(test_df[['brand_freq']].astype(float).values)
X_test_all = hstack([X_test_text, csr_matrix(X_test_num), csr_matrix(X_test_brand)], format='csr')


# Predict on test set
y_test_log_pred = model.predict(X_test_all)
y_test_pred = np.expm1(y_test_log_pred)
y_test_pred = np.clip(y_test_pred, 0.01, None) # Clip predictions to be at least 0.01

# Create submission file
out = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': y_test_pred})
out.to_csv('sample_model_test_out.csv', index=False)

print("Saved sample_model_test_out.csv")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Saved sample_model_test_out.csv
